In [ ]:
import nltk
import spacy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import DefaultDict, List, Tuple, Set, Hashable, Iterable, Dict
from collections import Counter, defaultdict
from tqdm import tqdm
from nltk import (
  CFG,  # Context-Free Grammar 
  PCFG  # Probabilistic Context-Free Grammar
)
from nltk.parse import (
  ChartParser,    # Chart Parsing Algorithm
  ViterbiParser   # Viterbi Parsing Algorithm
)
from nltk.tokenize import (
  word_tokenize,  # Tokenize a sentence into words
  sent_tokenize,  # Tokenize text into sentences
)
from spacy import displacy

try:
  nltk.data.find('tokenizers/punkt')
except LookupError:
  nltk.download('punkt')

nlp = spacy.load("es_core_news_sm")

# Tokenización
**TODO**: Integración del dataset en el notebook

In [ ]:
tokens_by_sent = {}
for i,sent in enumerate(sentences):
  clean_sent = sent.lower()
  tokens_by_sent[i] = word_tokenize(clean_sent, language='spanish')

for idx in tokens_by_sent.keys():
  print(f"Doc: {idx+1}: {sentences[idx]}")
  print(f"Tokens: {tokens_by_sent[idx]}")

# Procesar Textos con Spacy

In [ ]:
def process_spacy(texts, batch_size=64):
  return nlp.pipe(texts, batch_size=batch_size)
docs = [doc for doc in tqdm(process_spacy(sentences, batch_size=64), total=len(sentences))]

**Leyenda**:
- `NOUN`: Sustantivo
- `VERB`: Verbo
- `ADJ`: Adjetivo
- `ADV`: Adverbio
- `PROPN`: Nombre Propio
- `DET`: Determinante
- `PRON`: Pronombre
- `ADP`: Preposición
- `CCONJ`: Conjunción
- `SCONJ`: Conjunción Sub
- `INTJ`: Interjección
- `NUM`: Número

In [ ]:
categories = defaultdict(set)

tags = ["NOUN", "VERB", "AUX", "ADJ", "ADV", "PROPN", "DET", "PRON", "ADP", "CCONJ", "SCONJ", "INTJ", "NUM", "PUNCT"]
# Puede extenderse con: `NOUN__Gender=Masc|Number=Sing`, `NOUN__Gender=Fem|Number=Sing`, `VERB__Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin` 

for doc in docs:
  for token in doc:
    is_found = False
    for tag in tags:
      if token.pos_ == tag:
        categories[tag].add(token.text.lower())
        is_found = True
    if not is_found:
      print(f"Dont Found: {token.text} | {token.pos_}")
      # categories["OTHER"].add(token.text.lower())

for idx in categories.keys():
  print(f"TAG({idx}) = {categories[idx]}")

In [ ]:
from typing import Hashable


def invert_defaultdict_of_sets(d: DefaultDict[Hashable, Set[Hashable]]) -> DefaultDict[Hashable, Set[Hashable]]:
  inv: DefaultDict[Hashable, Set[Hashable]] = defaultdict(set)
  for k, vs in d.items():
    for v in vs:
      inv[v].add(k)
  return inv

def invert_defaultdict_of_lists(d: DefaultDict[Hashable, List[Hashable]]) -> DefaultDict[Hashable, List[Hashable]]:
  inv: DefaultDict[Hashable, List[Hashable]] = defaultdict(list)
  for k, vs in d.items():
    for v in vs:
      inv[v].append(k)
  return inv

def invert_dict(d: Dict[Hashable, Hashable]) -> Dict[Hashable, Hashable]:
  return {v: k for k, v in d.items()}

def invert_dict_multi(d: Dict[Hashable, Iterable[Hashable]]) -> DefaultDict[Hashable, Set[Hashable]]:
  inv: DefaultDict[Hashable, Set[Hashable]] = defaultdict(set)
  for k, vs in d.items():
    for v in vs:
      inv[v].add(k)
  return inv

inverse_categories = invert_defaultdict_of_sets(categories)
for idx in inverse_categories.keys():
  inverse_categories[idx] = list[Hashable](inverse_categories[idx])[0]
  print(f"WORD({idx}) = TAG({ inverse_categories[idx] })")

In [ ]:
for idx in tokens_by_sent.keys():
  print(f"Doc: {idx+1}: {sentences[idx]}")
  print([ inverse_categories[token] for token in tokens_by_sent[idx] ])

# Construcción de Reglas CFG

In [ ]:
S1 = """ 
S -> PRELUDE PUNCT CLAUSE PUNCT CLAUSE
PRELUDE -> SCONJ NP
CLAUSE -> NP VP
NP -> NP ADP NP CCONJ ADP NP
NP -> DET NOUN | DET NOUN ADJ
NP -> NOUN | PRON | PROPN 
VP -> VERB
VP -> AUX ADV ADJ
"""

S2 = """ 
S2 -> CLAUSE PUNCT CLAUSE PUNCT CLAUSE PUNCT CLAUSE

CLAUSE -> NP VP | VP NP
CLAUSE -> NP SCONJ VP

NP -> PROPN | PROPN NP
NP -> NOUN | NOUN NP   
NP -> DET NOUN ADJ

VP -> VERB ADP PRON SCONJ VERB
VP -> VERB | VERB ADP ADJ
VP -> ADV VERB NP
"""

S3 = """ 
S3 -> NP
NP -> PROPN | PROPN PP | PROPN NP PUNCT NP | PROPN NP CCONJ NP
NP -> PP
NP -> DET NP 
NP -> NOUN | NOUN ADJ 
PP -> ADP NP | ADP NP PP
"""

S4 = """ 
S4 -> CLAUSE SCONJ NP CLAUSE
CLAUSE -> ADP NP | ADV VERB SCONJ VERB
NP -> NOUN ADJ
NP -> NOUN PUNCT | NOUN PUNCT NP | NOUN
"""

S5 = """ 
S -> NP PP PUNCT PP PUNCT
NP -> PRON ADJ
PP -> ADP NPP | ADP NOUN 
NPP -> PROPN PROPN
PUNCT -> COMMA
"""

In [ ]:
def build_cfg(categories, ind_grammar, head_grammar):
  lexical_rules = []
  for tag, items in categories.items():
    if items:
      alts = " | ".join(sorted({f"'{w}'" for w in items}))
      lexical_rules.append(f"{tag} -> {alts}")
  
  return "\n".join(head_grammar + ind_grammar + lexical_rules)


head_grammar = ["S -> " + " | ".join([f"S{i+1}" for i in range(len(sentences))])]
ind_grammar = [S1, S2, S3, S4, S5]

grammar_text = build_cfg(categories, ind_grammar, head_grammar)
grammar = CFG.fromstring(grammar_text)
print(grammar)

In [ ]:
parser = ChartParser(grammar)
parsed = []
for i in range(len(sentences)):
  toks = tokens_by_sent[i]
  trees = list(parser.parse(toks))
  print(f"Sentence {i+1} parses: {len(trees)}")
  parsed.append(trees)

**Análisis Estadístico y Características Propuestas**:
1. Métricas de ambigüedad gramatical:
    1. Número de árboles de parsing por oración.
    2. Entropía de parsing (diversidad estructural).
    3. Ratio de ambigüedad normalizado por longitud.
2. Características estructurales:
    1. Profundidad máxima del árbol sintáctico.
    2. Ancho promedio/máximo del árbol
    3. Distribución de reglas aplicadas 
    4. Ratio de cláusulas subordinadas vs. principales.
3. Características léxicas basadas en POS tags:
    1. Frecuencia de cada categoría gramatical
    2. Ratio ADJ/NOUN, ADV/VERB
    3. Densidad léxica (content words / total words)
    4. TTR (Type-Token Ratio) por categoría
4. Características de complejidad:
    1. Longitud promedio de sintagmas nominales/verbales
    2. Número de coordinaciones y subordinaciones
    3. Índice de complejidad sintáctica

### Funciones para Extraer Características de Árboles Sintácticos

In [ ]:
def extract_tree_features(tree):
  "Extrae características estructurales de un árbol de parsing."
  features = {}

  # Profundidad del árbol
  features['tree_depth'] = tree.height()

  # Número de nodos
  features['num_nodes'] = len(list(tree.subtrees()))

  # Número de hojas (palabras)
  features['num_leaves'] = len(tree.leaves())

  # Ancho del árbol (máximo número de hijos en un nivel)
  def max_width(t):
    if isinstance(t, str):
      return 1
    widths = [max_width(child) for child in t]
    return max(len(t), max(widths) if widths else 1)

  features['tree_width'] = max_width(tree)

  # Factor de ramificación promedio
  if features['num_nodes'] > 1:
    features['branching_factor'] = (
        features['num_nodes'] - 1) / (features['num_nodes'] - features['num_leaves'])
  else:
    features['branching_factor'] = 0

  # Conteo de reglas aplicadas
  rules_counter = Counter()
  for subtree in tree.subtrees():
    if not isinstance(subtree, str) and len(subtree) > 0:
      rule = f"{subtree.label()} -> {' '.join([child.label() if hasattr(child, 'label') else str(child) for child in subtree])}"
      rules_counter[rule] += 1

  features['unique_rules'] = len(rules_counter)
  features['total_rules'] = sum(rules_counter.values())

  # Conteo de categorías sintácticas
  category_counter = Counter()
  for subtree in tree.subtrees():
    if hasattr(subtree, 'label'):
      category_counter[subtree.label()] += 1

  features['categories'] = dict(category_counter)

  return features

def extract_pos_features(tokens, inverse_categories):
  "Extrae características basadas en POS tags."
  features = {}

  # Obtener POS tags
  pos_tags = [inverse_categories.get(token, 'UNK') for token in tokens]
  pos_counter = Counter(pos_tags)

  # Frecuencias básicas
  total_tokens = len(tokens)
  features['total_tokens'] = total_tokens

  # Content words vs function words
  content_tags = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}
  function_tags = {'DET', 'ADP', 'CCONJ', 'SCONJ', 'PRON', 'AUX'}

  content_count = sum(pos_counter[tag] for tag in content_tags if tag in pos_counter)
  function_count = sum(pos_counter[tag] for tag in function_tags if tag in pos_counter)

  features['content_words'] = content_count
  features['function_words'] = function_count
  features['lexical_density'] = content_count / total_tokens if total_tokens > 0 else 0

  # Ratios específicos
  features['noun_count'] = pos_counter.get('NOUN', 0) + pos_counter.get('PROPN', 0)
  features['verb_count'] = pos_counter.get('VERB', 0) + pos_counter.get('AUX', 0)
  features['adj_count'] = pos_counter.get('ADJ', 0)
  features['adv_count'] = pos_counter.get('ADV', 0)

  features['adj_noun_ratio'] = features['adj_count'] / \
      features['noun_count'] if features['noun_count'] > 0 else 0
  features['adv_verb_ratio'] = features['adv_count'] / \
      features['verb_count'] if features['verb_count'] > 0 else 0
  features['noun_verb_ratio'] = features['noun_count'] / \
      features['verb_count'] if features['verb_count'] > 0 else 0

  # Type-Token Ratio (diversidad léxica)
  unique_tokens = len(set(tokens))
  features['ttr'] = unique_tokens / total_tokens if total_tokens > 0 else 0

  # Subordinación
  features['subordination_count'] = pos_counter.get('SCONJ', 0)
  features['coordination_count'] = pos_counter.get('CCONJ', 0)

  return features

def calculate_ambiguity_metrics(parse_trees, tokens):
  "Calcula métricas de ambigüedad gramatical"
  metrics = {}

  num_parses = len(parse_trees)
  metrics['num_parses'] = num_parses
  metrics['num_tokens'] = len(tokens)

  # Ratio de ambigüedad normalizado
  metrics['ambiguity_ratio'] = num_parses / len(tokens) if len(tokens) > 0 else 0

  # Si hay múltiples parses, calcular diversidad estructural
  if num_parses > 1:
    depths = [tree.height() for tree in parse_trees]
    metrics['parse_depth_variance'] = np.var(depths)
    metrics['parse_depth_mean'] = np.mean(depths)
    metrics['parse_depth_std'] = np.std(depths)
  else:
    metrics['parse_depth_variance'] = 0
    metrics['parse_depth_mean'] = parse_trees[0].height() if parse_trees else 0
    metrics['parse_depth_std'] = 0

  # Entropía de parsing (si hay múltiples parses, asumimos igual probabilidad)
  if num_parses > 1:
    metrics['parse_entropy'] = np.log2(num_parses)
  else:
    metrics['parse_entropy'] = 0

  return metrics



### Extraer todas las características del corpus

In [ ]:

def extract_all_features(sentences, tokens_by_sent, parsed, inverse_categories):
  "Extrae todas las características para cada oración"
  all_features = []

  for i in range(len(sentences)):
    sent_features = {
        'sentence_id': i + 1,
        'sentence': sentences[i],
        'tokens': tokens_by_sent[i]
    }

    # Características de ambigüedad
    ambiguity_metrics = calculate_ambiguity_metrics(parsed[i], tokens_by_sent[i])
    sent_features.update(ambiguity_metrics)

    # Características POS
    pos_features = extract_pos_features(tokens_by_sent[i], inverse_categories)
    sent_features.update(pos_features)

    # Características del primer árbol de parsing (o promedio si hay múltiples)
    if parsed[i]:
      tree_features = extract_tree_features(parsed[i][0])
      sent_features.update(tree_features)

      # Si hay múltiples parses, calcular promedios
      if len(parsed[i]) > 1:
        all_tree_features = [extract_tree_features(tree) for tree in parsed[i]]

        numeric_keys = ['tree_depth', 'num_nodes', 'num_leaves', 'tree_width',
                        'branching_factor', 'unique_rules', 'total_rules']

        for key in numeric_keys:
          values = [tf[key] for tf in all_tree_features]
          sent_features[f'{key}_mean'] = np.mean(values)
          sent_features[f'{key}_std'] = np.std(values)

    all_features.append(sent_features)

  return all_features


# Ejecutar extracción
features_list = extract_all_features(sentences, tokens_by_sent, parsed, inverse_categories)

print("Características extraídas para", len(features_list), "oraciones")
print("\nEjemplo de features de la primera oración:")
for key, value in list(features_list[0].items())[:15]:
  print(f"  {key}: {value}")



### Crear DataFrame y Análisis Estadístico

In [ ]:
# Crear DataFrame (excluyendo campos anidados como 'categories' y 'tokens')
def flatten_features(features_list):
  "Aplana las características para crear DataFrame"
  flattened = []
  for feat in features_list:
    flat_feat = {}
    for key, value in feat.items():
      if key not in ['categories', 'tokens']:  # Excluir campos complejos
        flat_feat[key] = value
    flattened.append(flat_feat)
  return flattened


flat_features = flatten_features(features_list)
df_features = pd.DataFrame(flat_features)

print("Shape del DataFrame:", df_features.shape)
print("\nPrimeras filas:")
display(df_features.head())

print("\nEstadísticas descriptivas:")
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
display(df_features[numeric_cols].describe())



### Visualizaciones

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# 1. Distribución de número de parses
axes[0, 0].bar(df_features['sentence_id'], df_features['num_parses'], color='steelblue')
axes[0, 0].set_xlabel('Oración')
axes[0, 0].set_ylabel('Número de Parses')
axes[0, 0].set_title('Ambigüedad Gramatical por Oración')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Profundidad del árbol vs longitud de oración
axes[0, 1].scatter(df_features['num_tokens'], df_features['tree_depth'],
                   s=100, alpha=0.6, color='coral')
axes[0, 1].set_xlabel('Número de Tokens')
axes[0, 1].set_ylabel('Profundidad del Árbol')
axes[0, 1].set_title('Complejidad Estructural')
axes[0, 1].grid(alpha=0.3)

# 3. Densidad léxica
axes[1, 0].bar(df_features['sentence_id'],
               df_features['lexical_density'], color='seagreen')
axes[1, 0].set_xlabel('Oración')
axes[1, 0].set_ylabel('Densidad Léxica')
axes[1, 0].set_title('Densidad Léxica (Content Words / Total)')
axes[1, 0].axhline(y=df_features['lexical_density'].mean(), color='red',
                   linestyle='--', label='Media')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Ratios gramaticales
x = np.arange(len(df_features))
width = 0.25
axes[1, 1].bar(x - width, df_features['adj_noun_ratio'],
               width, label='ADJ/NOUN', alpha=0.8)
axes[1, 1].bar(x, df_features['adv_verb_ratio'], width, label='ADV/VERB', alpha=0.8)
axes[1, 1].bar(x + width, df_features['noun_verb_ratio'],
               width, label='NOUN/VERB', alpha=0.8)
axes[1, 1].set_xlabel('Oración')
axes[1, 1].set_ylabel('Ratio')
axes[1, 1].set_title('Ratios Gramaticales')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

# 5. Distribución de tipos de palabras
word_types = df_features[['content_words', 'function_words']].sum()
axes[2, 0].pie(word_types, labels=['Content Words', 'Function Words'],
               autopct='%1.1f%%', startangle=90, colors=['lightblue', 'lightcoral'])
axes[2, 0].set_title('Distribución Global: Content vs Function Words')

# 6. Subordinación y coordinación
axes[2, 1].bar(df_features['sentence_id'], df_features['subordination_count'],
               label='Subordinación', alpha=0.7)
axes[2, 1].bar(df_features['sentence_id'], df_features['coordination_count'],
               label='Coordinación', alpha=0.7, bottom=df_features['subordination_count'])
axes[2, 1].set_xlabel('Oración')
axes[2, 1].set_ylabel('Conteo')
axes[2, 1].set_title('Complejidad Sintáctica')
axes[2, 1].legend()
axes[2, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()



### Matriz de Correlación

In [ ]:


# Seleccionar columnas numéricas relevantes para ML
ml_features = [
    'num_parses', 'ambiguity_ratio', 'parse_entropy',
    'total_tokens', 'lexical_density', 'ttr',
    'noun_count', 'verb_count', 'adj_count', 'adv_count',
    'adj_noun_ratio', 'adv_verb_ratio', 'noun_verb_ratio',
    'subordination_count', 'coordination_count',
    'tree_depth', 'num_nodes', 'branching_factor'
]

# Filtrar solo las que existen
ml_features = [f for f in ml_features if f in df_features.columns]

# Matriz de correlación
corr_matrix = df_features[ml_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=1)
plt.title('Matriz de Correlación de Features Gramaticales', fontsize=14)
plt.tight_layout()
plt.show()



### Análisis de relevancia para ML

In [ ]:

print("=" * 70)
print("ANÁLISIS DE RELEVANCIA PARA MACHINE LEARNING")
print("=" * 70)

print("\n1. FEATURES DE AMBIGÜEDAD:")
print("-" * 50)
ambiguity_features = ['num_parses', 'ambiguity_ratio', 'parse_entropy']
for feat in ambiguity_features:
  if feat in df_features.columns:
    print(
        f"  {feat:20s}: μ={df_features[feat].mean():.3f}, σ={df_features[feat].std():.3f}")

print("\n2. FEATURES DE COMPLEJIDAD ESTRUCTURAL:")
print("-" * 50)
struct_features = ['tree_depth', 'num_nodes', 'branching_factor', 'tree_width']
for feat in struct_features:
  if feat in df_features.columns:
    print(
        f"  {feat:20s}: μ={df_features[feat].mean():.3f}, σ={df_features[feat].std():.3f}")

print("\n3. FEATURES LÉXICAS:")
print("-" * 50)
lex_features = ['lexical_density', 'ttr', 'adj_noun_ratio', 'noun_verb_ratio']
for feat in lex_features:
  if feat in df_features.columns:
    print(
        f"  {feat:20s}: μ={df_features[feat].mean():.3f}, σ={df_features[feat].std():.3f}")

print("\n4. VARIANZA DE FEATURES:")
print("-" * 50)
variances = df_features[ml_features].var().sort_values(ascending=False)
print(variances.head(10))

print("\n" + "=" * 70)
print("RECOMENDACIONES PARA ML:")
print("=" * 70)
print("""
1. Features más prometedoras (alta varianza):
   - Longitud y conteos absolutos (num_tokens, noun_count, etc.)
   - Ambigüedad gramatical (num_parses, parse_entropy)
   - Complejidad estructural (tree_depth, num_nodes)

2. Features que requieren más datos:
   - Ratios gramaticales (pueden ser ruidosos con corpus pequeño)
   - TTR (sensible al tamaño del texto)

3. Posibles aplicaciones:
   - Clasificación de complejidad textual
   - Detección de estilo de escritura
   - Análisis de sentimientos (tu caso: reviews negativas)
   - Predicción de dificultad de lectura

4. Próximos pasos:
   - Expandir corpus para análisis robusto
   - Probar PCA/feature selection
   - Entrenar modelos de clasificación
   - Comparar con features de embeddings (BERT, etc.)
""")


In [ ]:
save = False 
if save:
  # Guardar features a CSV
  df_features.to_csv('grammar_features.csv', index=False)
  print("Features guardadas en 'grammar_features.csv'")